In [ ]:
import pandas as pd
test = pd.read_csv("amazon.csv")
test.head()

In [ ]:
train = pd.read_csv("amazon.csv")
train.head()

In [ ]:
# Menampilkan ringkasan informasi dari dataset
train.info()

In [ ]:
# Menampilkan statistik deskriptif dari dataset
train.describe(include="all")

In [ ]:
# Memeriksa jumlah nilai yang hilang di setiap kolom
missing_values = train.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
less = missing_values[missing_values < 1000].index
over = missing_values[missing_values >= 1000].index

In [ ]:
# Contoh mengisi nilai yang hilang dengan median untuk kolom numerik
numeric_features = train[less].select_dtypes(include=['number']).columns
train[numeric_features] = train[numeric_features].fillna(train[numeric_features].median())

In [ ]:
# Mengisi missing values dengan mode untuk kolom kategori
categorical_features = train[less].select_dtypes(
    include=['object']
).columns

for column in categorical_features:
    mode_value = train[column].mode()

    if not mode_value.empty:
        train[column] = train[column].fillna(mode_value[0])

In [ ]:
# Menghapus kolom dengan terlalu banyak nilai yang hilang
df = train.drop(columns=over)

In [ ]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

for feature in numeric_features:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[feature])
    plt.title(f'Box Plot of {feature}')
    plt.show()

In [ ]:
# Contoh sederhana untuk mengidentifikasi outliers menggunakan IQR
Q1 = df[numeric_features].quantile(0.25)
Q3 = df[numeric_features].quantile(0.75)
IQR = Q3 - Q1

In [ ]:
# Filter DataFrame untuk menghapus baris yang mengandung outlier
condition = ~(
    (df[numeric_features] < (Q1 - 1.5 * IQR)) |
    (df[numeric_features] > (Q3 + 1.5 * IQR))
).any(axis=1)

df_filtered_numeric = df.loc[condition, numeric_features]

# Menggabungkan kembali dengan kolom kategorikal
categorical_features = df.select_dtypes(include=['object', 'str']).columns

df = pd.concat(
    [df_filtered_numeric, df.loc[condition, categorical_features]],
    axis=1
)

In [ ]:
print(df.dtypes)

In [ ]:

# Kolom yang seharusnya berisi angka
kolom_angka = [
    'discounted_price',
    'actual_price',
    'discount_percentage',
    'rating',
    'rating_count'
]

# Mengubah teks menjadi angka
for kolom in kolom_angka:
    df[kolom] = (
        df[kolom].astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('₹', '', regex=False)
        .str.replace('%', '', regex=False)
        .str.strip()
    )

    df[kolom] = pd.to_numeric(df[kolom], errors='coerce')

# Memperbarui daftar fitur numerik
numeric_features = df.select_dtypes(
    include=['number']
).columns.tolist()

print("Kolom numerik:", numeric_features)
print(df[numeric_features].dtypes)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Memperbarui daftar kolom numerik
numeric_features = df.select_dtypes(include=['number']).columns.tolist()

# Mengecek kolom numerik
print("Kolom numerik:", numeric_features)
print("Ukuran data:", df.shape)

# Standardisasi fitur numerik
if numeric_features and not df.empty:
    scaler = StandardScaler()
    df[numeric_features] = scaler.fit_transform(df[numeric_features])
    print("Standardisasi berhasil!")
else:
    print("Data kosong atau tidak ada kolom numerik.")

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

feature = numeric_features[0]

plt.figure(figsize=(12, 5))
sns.histplot(data=df, x=feature, kde=True)
plt.title(f"Histogram Sebelum Standardisasi - {feature}")
plt.show()


In [ ]:
# Histogram Sebelum Standardisasi
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(train[numeric_features[0]], kde=True)
plt.title("Histogram Sebelum Standardisasi")

plt.tight_layout()
plt.show()

In [ ]:
# Histogram Setelah Standardisasi
plt.subplot(1, 2, 2)
sns.histplot(df[numeric_features[3]], kde=True)
plt.title("Histogram Setelah Standardisasi")

In [ ]:
# Mengidentifikasi baris duplikat
duplicates = df.duplicated()

print("Baris duplikat:")
print(df[duplicates])

In [ ]:
# Menghapus baris duplikat
df = df.drop_duplicates()

print("DataFrame setelah menghapus duplikat:")
print(df)

In [ ]:
category_features = df.select_dtypes(include=['object']).columns
df[category_features]

In [ ]:
df_one_hot = pd.get_dummies(df, columns=category_features)
df_one_hot

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Inisialisasi LabelEncoder
label_encoder = LabelEncoder()
df_lencoder = pd.DataFrame(df)

# Mengubah kolom kategorikal menjadi angka
for col in category_features:
    df_lencoder[col] = label_encoder.fit_transform(
        df[col].astype(str)
    )

# Menampilkan hasil
df_lencoder

In [ ]:
df_lencoder.head()

In [ ]:
# Menghitung jumlah dan persentase missing values
missing_values = df_lencoder.isnull().sum()
missing_percentage = (missing_values / len(df_lencoder)) * 100

# Membuat DataFrame hasil pemeriksaan
missing_data = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
}).sort_values(by='Missing Values', ascending=False)

# Menampilkan kolom yang memiliki missing values
missing_data[missing_data['Missing Values'] > 0]

In [ ]:

# Menghitung jumlah variabel
num_vars = df_lencoder.shape[1]

# Menentukan jumlah baris dan kolom subplot
n_cols = 4
n_rows = -(-num_vars // n_cols)

# Membuat subplot
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(20, n_rows * 4)
)

# Meratakan array axes
axes = axes.flatten()

# Plot setiap variabel
for i, column in enumerate(df_lencoder.columns):
    df_lencoder[column].hist(
        ax=axes[i],
        bins=20,
        edgecolor='black'
    )
    axes[i].set_title(column)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

# Menghapus subplot yang tidak terpakai
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Merapikan layout
plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Kolom dataset Amazon yang ingin divisualisasikan
columns_to_plot = [
    'discounted_price',
    'actual_price',
    'discount_percentage',
    'rating',
    'rating_count'
]

plt.figure(figsize=(15, 10))

for i, column in enumerate(columns_to_plot, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[column].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {column}')

plt.tight_layout()
plt.show()


In [ ]:

# Visualisasi korelasi antar variabel numerik
import matplotlib.pyplot as plt
import seaborn as sns

numeric_features = df.select_dtypes(include=['number']).columns.tolist()

plt.figure(figsize=(12, 10))
correlation_matrix = df[numeric_features].corr()

sns.heatmap(
    correlation_matrix,
    annot=False,
    cmap='coolwarm',
    vmin=-1,
    vmax=1
)

plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()


In [ ]:

# Visualisasi korelasi antar variabel numerik
import matplotlib.pyplot as plt
import seaborn as sns

numeric_features = df.select_dtypes(include=['number']).columns.tolist()

plt.figure(figsize=(12, 10))
correlation_matrix = df[numeric_features].corr()

sns.heatmap(
    correlation_matrix,
    annot=False,
    cmap='coolwarm',
    vmin=-1,
    vmax=1
)

plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()


In [ ]:
import sklearn

In [ ]:

# Memisahkan fitur (X) dan target (y)
X = df.drop(columns=['rating'])
y = df['rating']

print("Jumlah data:", len(X))
print("Jumlah fitur:", X.shape[1])
print("Jumlah target:", len(y))


In [ ]:

from sklearn.model_selection import train_test_split

# Membagi dataset menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1
)

print("Jumlah data:", len(X))
print("Jumlah data latih:", len(X_train))
print("Jumlah data test:", len(X_test))


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Memilih fitur dan target
fitur = [
    'discounted_price',
    'actual_price',
    'discount_percentage',
    'rating_count'
]

data_model = df[fitur + ['rating']].dropna()

X = data_model[fitur]
y = data_model['rating']

# Membagi data training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Membuat dan melatih model Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Melakukan prediksi
y_pred = model.predict(X_test)

# Evaluasi model
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Memilih fitur dan target
fitur = [
    'discounted_price',
    'actual_price',
    'discount_percentage',
    'rating_count'
]

data_model = df[fitur + ['rating']].dropna()

X = data_model[fitur]
y = data_model['rating']

# Membagi data training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Melatih model Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

# Prediksi dan evaluasi
y_pred = rf_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("R2 Score:", r2_score(y_test, y_pred))